In [1]:
# !pip install torch torchvision torchaudio
# !pip install transformers peft accelerate tqdm

In [ ]:
# THIS SEEMS TO WORK BETTER
# !pip uninstall -y transformers peft accelerate torch torchvision torchaudio

# !pip install \
#   torch==2.3.1 \
#   torchvision==0.18.1 \
#   torchaudio==2.3.1

# !pip install \
#   transformers==4.45.2 \
#   peft==0.13.2 \
#   accelerate==0.34.2 \
#   tqdm

In [ ]:
import json
import pickle
import os
from typing import Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from tqdm import tqdm

# ─────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────

MODEL_NAME       = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_FILE       = "train.jsonl"
TEST_FILE        = "test.jsonl"
OUTPUT_DIR       = "dual_lora_output"
PRED_OUTPUT_FILE = "preds.pkl"

# Asymmetric ranks — keep alpha/r = 2.0 for both
LORA_R_P     = 12    # prompt adapter rank
LORA_ALPHA_P = 24    # prompt adapter alpha
LORA_R_C     = 4   # completion adapter rank
LORA_ALPHA_C = 8   # completion adapter alpha
LORA_DROPOUT = 0.05

# Parameter-matched single LoRA: r = LORA_R_P + LORA_R_C = 16, alpha = 32
# Architecture-matched (half params): r = 8, alpha = 16

TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

EPOCHS       = 3
BATCH_SIZE   = 4
GRAD_ACCUM   = 4
LR           = 2e-4
WARMUP_RATIO = 0.05
MAX_SEQ_LEN  = 512
DTYPE        = torch.bfloat16

MAX_NEW_TOKENS = 256
DO_SAMPLE      = False

# ─────────────────────────────────────────────
#  DUAL-ADAPTER LINEAR LAYER
# ─────────────────────────────────────────────

class DualLoRALinear(nn.Module):
    """
    Frozen base Linear + two LoRA adapter pairs with independent ranks.

    lora_P: in_f -> r_P -> out_f,  scaling = alpha_P / r_P
    lora_C: in_f -> r_C -> out_f,  scaling = alpha_C / r_C

    _seq_len controls routing:
      _seq_len ==  0  prefill  -> lora_P only (inference, prompt)
      _seq_len ==  1  decode   -> lora_C only (inference, generation)
      _seq_len >   1  train    -> blended via role_mask
      _seq_len == -1  unset    -> fallback lora_C (should not occur)
    """

    def __init__(self, base_linear: nn.Linear,
                 r_P: int, alpha_P: float,
                 r_C: int, alpha_C: float,
                 dropout: float):
        super().__init__()
        self.base      = base_linear
        in_f           = base_linear.in_features
        out_f          = base_linear.out_features
        self.scaling_P = alpha_P / r_P
        self.scaling_C = alpha_C / r_C

        dev  = next(base_linear.parameters()).device
        dtyp = next(base_linear.parameters()).dtype

        self.lora_P_A = nn.Linear(in_f, r_P,  bias=False, device=dev, dtype=dtyp)
        self.lora_P_B = nn.Linear(r_P,  out_f, bias=False, device=dev, dtype=dtyp)
        self.lora_C_A = nn.Linear(in_f, r_C,  bias=False, device=dev, dtype=dtyp)
        self.lora_C_B = nn.Linear(r_C,  out_f, bias=False, device=dev, dtype=dtyp)
        self.dropout  = nn.Dropout(dropout)

        nn.init.kaiming_uniform_(self.lora_P_A.weight, a=5**0.5)
        nn.init.zeros_(self.lora_P_B.weight)
        nn.init.kaiming_uniform_(self.lora_C_A.weight, a=5**0.5)
        nn.init.zeros_(self.lora_C_B.weight)

        for p in self.base.parameters():
            p.requires_grad = False

        self.role_mask: Optional[torch.Tensor] = None
        self._seq_len:  int = -1

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base(x)

        if self._seq_len == 0:
            # Prefill: prompt tokens -> lora_P
            delta = self.lora_P_B(self.dropout(self.lora_P_A(x))) * self.scaling_P

        elif self._seq_len == 1:
            # Decode: one new token at a time -> lora_C
            delta = self.lora_C_B(self.dropout(self.lora_C_A(x))) * self.scaling_C

        elif self._seq_len > 1 and self.role_mask is not None and x.shape[1] == self._seq_len:
            # Training: blend by role_mask
            lora_P_out = self.lora_P_B(self.dropout(self.lora_P_A(x))) * self.scaling_P
            lora_C_out = self.lora_C_B(self.dropout(self.lora_C_A(x))) * self.scaling_C
            mask  = self.role_mask.unsqueeze(-1).to(x.dtype)
            delta = (1.0 - mask) * lora_P_out + mask * lora_C_out

        else:
            # Fallback
            delta = self.lora_C_B(self.dropout(self.lora_C_A(x))) * self.scaling_C

        return base_out + delta


# ─────────────────────────────────────────────
#  DUAL-ADAPTER MODEL WRAPPER
# ─────────────────────────────────────────────

class DualLoRAModel(nn.Module):

    def __init__(self, base_model, target_modules,
                 r_P, alpha_P, r_C, alpha_C, dropout):
        super().__init__()
        self.model = base_model
        self._dual_layers: list[DualLoRALinear] = []
        self._inject(target_modules, r_P, alpha_P, r_C, alpha_C, dropout)
        self._freeze_base()

    def _inject(self, target_modules, r_P, alpha_P, r_C, alpha_C, dropout):
        # Collect before mutating — avoids stale references from mid-iteration setattr
        replacements = []
        for name, module in self.model.named_modules():
            if not isinstance(module, nn.Linear):
                continue
            if not any(t in name for t in target_modules):
                continue
            replacements.append((name, module))

        for name, module in replacements:
            dual = DualLoRALinear(module, r_P, alpha_P, r_C, alpha_C, dropout)
            self._dual_layers.append(dual)
            parts  = name.split(".")
            parent = self.model
            for part in parts[:-1]:
                parent = getattr(parent, part)
            setattr(parent, parts[-1], dual)

    def _freeze_base(self):
        for name, param in self.named_parameters():
            param.requires_grad = ("lora_P" in name or "lora_C" in name)

    def _set_mode(self, mode: str, role_mask: Optional[torch.Tensor] = None,
                  seq_len: int = -1):
        for layer in self._dual_layers:
            if mode == "train":
                layer.role_mask = role_mask
                layer._seq_len  = seq_len
            elif mode == "prefill":
                layer.role_mask = None
                layer._seq_len  = 0
            elif mode == "decode":
                layer.role_mask = None
                layer._seq_len  = 1

    def _clear(self):
        for layer in self._dual_layers:
            layer.role_mask = None
            layer._seq_len  = -1

    # ── training forward ─────────────────────────────────────────────────────

    def forward(self, input_ids, attention_mask=None, labels=None,
                role_mask=None, **kwargs):
        if role_mask is not None:
            self._set_mode("train", role_mask=role_mask,
                           seq_len=input_ids.shape[1])
        out = self.model(input_ids=input_ids, attention_mask=attention_mask,
                         labels=labels, **kwargs)
        self._clear()
        return out

    # ── two-phase inference ───────────────────────────────────────────────────

    def generate_dual(self, input_ids, attention_mask, max_new_tokens,
                      pad_token_id, eos_token_id):
        """
        Phase 1 - Prefill (lora_P):
            Full prompt forward to populate KV cache. No token sampled.

        Phase 2 - Decode (lora_C):
            Token #1 sampled from prefill logits (KV cache already has full
            prompt, so no re-feed). All subsequent tokens generated
            autoregressively under lora_C.
        """
        device = input_ids.device

        # Phase 1: prefill with lora_P
        self._set_mode("prefill")
        with torch.no_grad():
            prefill_out = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=True,
            )
        past_kv = prefill_out.past_key_values

        # Phase 2: decode with lora_C
        self._set_mode("decode")
        next_token   = prefill_out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated    = [next_token]
        current_attn = torch.cat(
            [attention_mask,
             torch.ones(input_ids.shape[0], 1, device=device, dtype=attention_mask.dtype)],
            dim=1
        )

        with torch.no_grad():
            for _ in range(max_new_tokens - 1):
                out = self.model(
                    input_ids=next_token,
                    attention_mask=current_attn,
                    past_key_values=past_kv,
                    use_cache=True,
                )
                past_kv    = out.past_key_values
                next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                generated.append(next_token)
                current_attn = torch.cat(
                    [current_attn,
                     torch.ones(input_ids.shape[0], 1,
                                device=device, dtype=attention_mask.dtype)],
                    dim=1
                )
                if (next_token == eos_token_id).all():
                    break

        self._clear()
        return torch.cat(generated, dim=1)

    # ── persistence ──────────────────────────────────────────────────────────

    def save_adapters(self, path: str):
        os.makedirs(path, exist_ok=True)
        state = {k: v for k, v in self.state_dict().items()
                 if "lora_P" in k or "lora_C" in k}
        torch.save(state, os.path.join(path, "dual_lora_adapters.pt"))
        print(f"Saved {len(state)} adapter tensors -> {path}/dual_lora_adapters.pt")

    def load_adapters(self, path: str):
        state = torch.load(os.path.join(path, "dual_lora_adapters.pt"),
                           map_location="cpu")
        missing, _ = self.load_state_dict(state, strict=False)
        lora_missing = [k for k in missing if "lora_P" in k or "lora_C" in k]
        print(f"Loaded {len(state)} adapter tensors. "
              f"LoRA keys missing (should be 0): {len(lora_missing)}")

    def trainable_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def total_parameters(self):
        return sum(p.numel() for p in self.parameters())


# ─────────────────────────────────────────────
#  DATASET
# ─────────────────────────────────────────────

class DualLoRADataset(Dataset):
    def __init__(self, filepath: str, tokenizer, max_len: int):
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.samples   = []
        with open(filepath) as f:
            for line in f:
                obj = json.loads(line.strip())
                self.samples.append((obj["prompt"], obj["completion"]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        prompt, completion = self.samples[idx]

        messages    = [{"role": "user", "content": prompt}]
        prompt_text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        prompt_ids = self.tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

        full_enc = self.tokenizer(
            prompt_text + completion,
            max_length=self.max_len,
            truncation=True,
            add_special_tokens=False,
        )
        input_ids  = full_enc["input_ids"]
        attn_mask  = full_enc["attention_mask"]
        prompt_len = min(len(prompt_ids), len(input_ids))

        labels = [-100] * prompt_len + input_ids[prompt_len:]

        if input_ids[-1] != self.tokenizer.eos_token_id:
            if len(input_ids) < self.max_len:
                input_ids.append(self.tokenizer.eos_token_id)
                attn_mask.append(1)
                labels.append(self.tokenizer.eos_token_id)

        labels    = labels[:len(input_ids)]
        role_mask = [0] * prompt_len + [1] * (len(input_ids) - prompt_len)
        role_mask = role_mask[:len(input_ids)]

        return {
            "input_ids":      torch.tensor(input_ids,  dtype=torch.long),
            "attention_mask": torch.tensor(attn_mask,  dtype=torch.long),
            "labels":         torch.tensor(labels,     dtype=torch.long),
            "role_mask":      torch.tensor(role_mask,  dtype=torch.bool),
        }


def collate_fn(batch, pad_id):
    max_len   = max(b["input_ids"].shape[0] for b in batch)
    input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    attn_mask = torch.zeros(len(batch), max_len,          dtype=torch.long)
    labels    = torch.full((len(batch), max_len), -100,   dtype=torch.long)
    role_mask = torch.zeros(len(batch), max_len,          dtype=torch.bool)

    for i, b in enumerate(batch):
        n = b["input_ids"].shape[0]
        input_ids[i, :n] = b["input_ids"]
        attn_mask[i, :n] = b["attention_mask"]
        labels[i, :n]    = b["labels"]
        role_mask[i, :n] = b["role_mask"]

    return {"input_ids": input_ids, "attention_mask": attn_mask,
            "labels": labels, "role_mask": role_mask}


# ─────────────────────────────────────────────
#  TRAINING
# ─────────────────────────────────────────────

def train():
    print("Loading tokenizer and base model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True
    )

    print("Injecting asymmetric dual LoRA adapters...")
    model = DualLoRAModel(
        base_model, TARGET_MODULES,
        LORA_R_P, LORA_ALPHA_P,
        LORA_R_C, LORA_ALPHA_C,
        LORA_DROPOUT,
    )
    total  = model.total_parameters()
    train_ = model.trainable_parameters()
    print(f"Total: {total/1e6:.1f}M | Trainable: {train_/1e6:.1f}M ({100*train_/total:.2f}%)")
    print(f"lora_P r={LORA_R_P} alpha={LORA_ALPHA_P} | lora_C r={LORA_R_C} alpha={LORA_ALPHA_C}")
    print(f"Parameter-matched single LoRA: r={LORA_R_P + LORA_R_C} alpha={(LORA_ALPHA_P + LORA_ALPHA_C)}")

    device  = next(model.parameters()).device
    dataset = DualLoRADataset(TRAIN_FILE, tokenizer, MAX_SEQ_LEN)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                         collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id))

    optimizer    = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01
    )
    total_steps  = (len(loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    print(f"Training {EPOCHS} epochs | {total_steps} optimizer steps")
    model.train()

    for epoch in range(EPOCHS):
        total_loss = 0.0
        optimizer.zero_grad()
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels    = batch["labels"].to(device)
            role_mask = batch["role_mask"].to(device)

            out  = model(input_ids=input_ids, attention_mask=attn_mask,
                         labels=labels, role_mask=role_mask)
            loss = out.loss / GRAD_ACCUM
            loss.backward()
            total_loss += loss.item() * GRAD_ACCUM

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0
                )
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                pbar.set_postfix(loss=f"{total_loss/(step+1):.4f}",
                                 lr=f"{scheduler.get_last_lr()[0]:.2e}")

        print(f"Epoch {epoch+1} avg loss: {total_loss/len(loader):.4f}")

    model.save_adapters(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Training complete.")


# ─────────────────────────────────────────────
#  INFERENCE
# ─────────────────────────────────────────────

def infer():
    print("Loading tokenizer and base model...")
    tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True
    )

    model = DualLoRAModel(
        base_model, TARGET_MODULES,
        LORA_R_P, LORA_ALPHA_P,
        LORA_R_C, LORA_ALPHA_C,
        LORA_DROPOUT,
    )
    model.load_adapters(OUTPUT_DIR)
    model.eval()

    device = next(model.parameters()).device

    with open(TEST_FILE) as f:
        lines = [json.loads(l.strip()) for l in f]

    results = []
    print(f"Generating on {len(lines)} test examples...")
    for i, obj in enumerate(tqdm(lines)):
        prompt      = obj["prompt"]
        messages    = [{"role": "user", "content": prompt}]
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        enc = tokenizer(prompt_text, return_tensors="pt",
                        add_special_tokens=False).to(device)

        gen_ids = model.generate_dual(
            input_ids      = enc["input_ids"],
            attention_mask = enc["attention_mask"],
            max_new_tokens = MAX_NEW_TOKENS,
            pad_token_id   = tokenizer.pad_token_id,
            eos_token_id   = tokenizer.eos_token_id,
        )

        prediction = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

        if i < 5:
            print(prediction)
        results.append(prediction)

        if i >= 499:
            break

    with open(PRED_OUTPUT_FILE, "wb") as f:
        pickle.dump(results, f)
    print(f"Saved {len(results)} predictions -> {PRED_OUTPUT_FILE}")

In [3]:
train()

Loading tokenizer and base model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Injecting asymmetric dual LoRA adapters...
Total: 3115.9M | Trainable: 29.9M (0.96%)
lora_P r=12 alpha=24 | lora_C r=4 alpha=8
Parameter-matched single LoRA: r=16 alpha=32
Training 3 epochs | 561 optimizer steps


Epoch 1/3: 100%|██████████| 750/750 [03:47<00:00,  3.29it/s, loss=0.2090, lr=1.59e-04]


Epoch 1 avg loss: 0.2085


Epoch 2/3: 100%|██████████| 750/750 [03:44<00:00,  3.34it/s, loss=0.0321, lr=5.48e-05]


Epoch 2 avg loss: 0.0321


Epoch 3/3: 100%|██████████| 750/750 [03:50<00:00,  3.26it/s, loss=0.0147, lr=0.00e+00]


Epoch 3 avg loss: 0.0147
Saved 1008 adapter tensors -> dual_lora_output/dual_lora_adapters.pt
Training complete.


In [5]:
infer()

Loading tokenizer and base model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/tmp/ipykernel_2508/2634442270.py:279: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(os.path.join(path, "dual_lora_adapters.pt"),


Loaded 1008 adapter tensors. LoRA keys missing (should be 0): 0
Generating on 4624 test examples...


  0%|          | 1/4624 [00:03<4:14:37,  3.30s/it]

...

[{'entity_type': 'PERSON', 'entity_span': 'Keith Olbermann'}, {'entity_type': 'ORG', 'entity_span': 'MSNBC'}]


  0%|          | 5/4624 [00:14<3:43:38,  2.91s/it]

[{'entity_type': 'ORG', 'entity_span': 'McDonald 's'}, {'entity_type': 'GPE', 'entity_span': 'US'}, {'entity_type': 'ORG', 'entity_span': 'McDonald 's'}, {'entity_type': 'GPE', 'entity_span': 'China'}]


 11%|█         | 499/4624 [18:04<2:29:25,  2.17s/it]

Saved 500 predictions -> preds.pkl
